<a href="https://colab.research.google.com/github/mahalaxmi-k/face-segregation-unsupervised-learning/blob/main/face_segregation_unsupervised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ================================
# INSTALL CELL — InsightFace + HDBSCAN (High-Accuracy Setup)
# ================================

# 1️⃣ Clean old/conflicting packages
!pip uninstall -y numpy torch torchvision torchaudio pillow opencv-python opencv-python-headless > /dev/null

# 2️⃣ Core scientific + ML stack (versions chosen for max compatibility)
# Install numpy and scipy/scikit-learn first with known compatible versions
!pip install -q numpy==1.26.0 pillow==10.3.0 torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 scikit-learn==1.4.2 tqdm==4.67.1 matplotlib==3.9.2

# 3️⃣ Face detection & recognition (removed retinaface)
!pip install -q insightface==0.7.3 onnxruntime-gpu==1.18.1 hdbscan==0.8.33

# 4️⃣ OpenCV (install after numpy)
!pip install -q opencv-python==4.9.0.80

# 5️⃣ Image / RAW / PSD / HEIF / TIFF handling
!pip install -q imageio==2.34.1 rawpy==0.25.1 pillow-heif==0.15.0 psd-tools==1.10.4 tifffile==2024.8.30

# 6️⃣ Optional utility packages
# scikit-image==0.21.1 requires numpy<2.0,>=1.22, so 1.26.0 should be fine
!pip install -q scikit-image==0.21.1


!echo "✅ All dependencies installed successfully!"
!echo "➡️  Please RESTART runtime once this finishes (Runtime > Restart runtime)."

In [ ]:
# --- CLEAN ENVIRONMENT ---
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless albumentations albucore hdbscan > /dev/null 2>&1
!pip cache purge
!rm -rf ~/.cache/pip ~/.cache/insightface ~/.insightface ~/.insightface_models /content/sample_data


In [ ]:
# --- CORE DEPENDENCIES ---
!pip install --upgrade pip setuptools wheel --quiet

# PyTorch (CUDA 12.1 build)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --quiet

# Stable versions tested on Colab (Python 3.12)
!pip install onnxruntime-gpu==1.19.2 insightface==0.7.3 hdbscan==0.8.40 tqdm matplotlib numpy==1.26.4 scipy==1.13.1 scikit-learn==1.5.2 scikit-image==0.22.0 --quiet

# Compatible OpenCV + Albumentations pair
!pip install opencv-python-headless==4.10.0.84 albumentations==1.4.4 --quiet


In [ ]:
import torch, onnxruntime, insightface, hdbscan, cv2, albumentations, numpy, tqdm, matplotlib

print(f"Torch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
print(f"ONNX Runtime: {onnxruntime.__version__} | Providers: {onnxruntime.get_available_providers()}")
print(f"InsightFace: {insightface.__version__}")

try:
    import importlib.metadata
    print("HDBSCAN:", importlib.metadata.version("hdbscan"))
except Exception:
    print("HDBSCAN: version check skipped")

print(f"OpenCV: {cv2.__version__}")
print(f"Albumentations: {albumentations.__version__}")
print("✅ All libraries imported successfully and ready to use!")



In [ ]:
# ============================================
# ⚙️ PRO-LEVEL FACE CLUSTERING PIPELINE
# ============================================

import os, cv2, torch, shutil, json, numpy as np
from pathlib import Path
from tqdm import tqdm
from insightface.app import FaceAnalysis
from PIL import Image, ImageDraw, ImageFont
from collections import defaultdict
import hdbscan
import math

# ---------- CONFIG ----------
INPUT_DIR = '/content/drive/MyDrive/MyDrive/input_photos'
OUTPUT_DIR = '/content/output_pro'
MAX_SIZE = 1600
MIN_FACE_SIZE = 80
MIN_DET_SCORE = 0.85
THUMB_SIZE = 160
PADDING = 5
SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)

# ---------- IMAGE PREPROCESSING ----------
def preprocess_image(img):
    """Resize + CLAHE grayscale + convert back to BGR"""
    h, w = img.shape[:2]
    if max(h, w) > MAX_SIZE:
        scale = MAX_SIZE / max(h, w)
        img = cv2.resize(img, (int(w*scale), int(h*scale)))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    gray = clahe.apply(gray)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

def list_images(folder):
    exts = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp'}
    return sorted([str(p) for p in Path(folder).rglob('*') if p.suffix.lower() in exts])

def read_image(path):
    img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Unable to read {path}")
    return preprocess_image(img)

# ---------- FACE MODEL ----------
print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
face_app = FaceAnalysis(
    name='buffalo_l',
    providers=['CUDAExecutionProvider' if torch.cuda.is_available() else 'CPUExecutionProvider']
)
face_app.prepare(ctx_id=0 if torch.cuda.is_available() else -1)

# ---------- DETECT FACES & EMBEDDINGS ----------
image_paths = list_images(INPUT_DIR)
all_embeddings, meta = [], []

for img_path in tqdm(image_paths, desc="Detecting Faces"):
    try:
        img = read_image(img_path)
        faces = face_app.get(img)
        if not faces:
            continue
        for f in faces:
            x1, y1, x2, y2 = [int(v) for v in f.bbox]
            if (x2-x1)<MIN_FACE_SIZE or (y2-y1)<MIN_FACE_SIZE:
                continue
            if hasattr(f,'det_score') and f.det_score<MIN_DET_SCORE:
                continue
            emb = f.embedding / np.linalg.norm(f.embedding)
            all_embeddings.append(emb.astype(np.float32))
            meta.append({
                'img': img_path,
                'bbox': [x1, y1, x2, y2],
                'embedding_idx': len(all_embeddings)-1
            })
    except Exception as e:
        print(f"❌ {img_path} {e}")

if not all_embeddings:
    raise SystemExit("No faces detected!")

embeddings = np.vstack(all_embeddings)
print(f"✅ Total faces detected: {len(embeddings)}")

# ---------- HDBSCAN CLUSTERING ----------
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=2,
    metric='euclidean', # Changed 'cosine' to 'euclidean'
    cluster_selection_method='eom'
)
labels = clusterer.fit_predict(embeddings)

# ---------- MAP CLUSTERS TO PERSONS ----------
unique_labels = [l for l in set(labels) if l != -1]
cluster_to_person = {str(lbl): f"person_{i+1:02d}" for i, lbl in enumerate(unique_labels)}
person_assignments = ['unknown' if l==-1 else cluster_to_person[str(l)] for l in labels]

# ---------- CREATE OUTPUT FOLDERS ----------
for pid in cluster_to_person.values():
    os.makedirs(os.path.join(OUTPUT_DIR, pid, 'solo'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, pid, 'group'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'unknown'), exist_ok=True)

# ---------- SOLO/GROUP ASSIGNMENT ----------
image_to_persons = defaultdict(list)
image_to_face_count = defaultdict(int)

for m, person in zip(meta, person_assignments):
    image_to_persons[m['img']].append(person)
    image_to_face_count[m['img']] += 1

for img, persons in tqdm(image_to_persons.items(), desc="Classifying Solo/Group"):
    face_count = image_to_face_count[img]
    unique_known = set([p for p in persons if p != 'unknown'])

    if face_count == 0 or not unique_known:
        shutil.copy2(img, os.path.join(OUTPUT_DIR, 'unknown', Path(img).name))
        continue

    if face_count == 1:
        solo_pid = list(unique_known)[0]
        shutil.copy2(img, os.path.join(OUTPUT_DIR, solo_pid, 'solo', Path(img).name))
        continue

    for pid in unique_known:
        shutil.copy2(img, os.path.join(OUTPUT_DIR, pid, 'group', Path(img).name))

print("✅ Solo/Group classification done!")

# ---------- SAVE METADATA ----------
persons_meta = {}
for pid in cluster_to_person.values():
    solo_dir = Path(OUTPUT_DIR)/pid/'solo'
    group_dir = Path(OUTPUT_DIR)/pid/'group'
    persons_meta[pid] = {
        'solo_images': sorted([p.name for p in solo_dir.glob('*')]),
        'group_images': sorted([p.name for p in group_dir.glob('*')])
    }
with open(Path(OUTPUT_DIR)/'persons_meta.json', 'w') as f:
    json.dump(persons_meta, f, indent=2)
print("✅ Metadata saved.")

# ---------- VISUAL REPORT WITH BOUNDING BOXES ----------
def display_thumbnail_grid(images, title="", thumb_size=THUMB_SIZE, padding=PADDING):
    if not images:
        print(f"No images for {title}")
        return
    n = len(images)
    cols = min(n, 5)
    rows = math.ceil(n / cols)
    grid_w = cols * thumb_size + (cols - 1) * padding
    grid_h = rows * thumb_size + (rows - 1) * padding
    grid_img = Image.new('RGB', (grid_w, grid_h), (255,255,255))
    for idx, img_path in enumerate(images):
        img = Image.open(img_path).convert('RGB')
        img.thumbnail((thumb_size, thumb_size))
        x = (idx % cols) * (thumb_size + padding)
        y = (idx // cols) * (thumb_size + padding)
        grid_img.paste(img, (x, y))
    print(f"🔹 {title} ({len(images)} images)")
    display(grid_img)

# Display thumbnails for each person
for pid, data in persons_meta.items():
    solo_paths = [Path(OUTPUT_DIR)/pid/'solo'/name for name in data['solo_images']]
    group_paths = [Path(OUTPUT_DIR)/pid/'group'/name for name in data['group_images']]

    display_thumbnail_grid(solo_paths, title=f"{pid} - SOLO")
    display_thumbnail_grid(group_paths, title=f"{pid} - GROUP")

In [ ]:
# ============================================
# 📊 CLUSTERING EVALUATION METRICS
# ============================================

from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import numpy as np

# Convert to numpy arrays
labels = np.array(labels)
embeddings = np.array(embeddings)

# Filter out noise (-1) for metric computations
mask = labels != -1
valid_embeddings = embeddings[mask]
valid_labels = labels[mask]

if len(set(valid_labels)) > 1 and len(valid_embeddings) > 10:
    sil_score = silhouette_score(valid_embeddings, valid_labels, metric='euclidean')
    ch_score = calinski_harabasz_score(valid_embeddings, valid_labels)
    db_score = davies_bouldin_score(valid_embeddings, valid_labels)

    print("==== 🧠 CLUSTERING QUALITY METRICS ====")
    print(f"Silhouette Score          : {sil_score:.4f}  (Higher = Better)")
    print(f"Calinski-Harabasz Index   : {ch_score:.4f}  (Higher = Better)")
    print(f"Davies-Bouldin Index      : {db_score:.4f}  (Lower = Better)")
    print(f"Detected Clusters (no noise): {len(set(valid_labels))}")
    print(f"Noise Faces (-1 label): {np.sum(labels==-1)}")
else:
    print("⚠ Not enough valid clusters for evaluation.")

In [ ]:
!pip install optuna

In [ ]:
# ============================================
# 🧠 BAYESIAN OPTIMIZATION — FULL HDBSCAN TUNING (FIXED DTYPE)
# ============================================

import optuna
import numpy as np
import hdbscan
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import pairwise_distances

# Ensure embeddings exist
if 'embeddings' not in locals():
    raise ValueError("❌ Embeddings not found. Run the main pipeline first.")

# Convert embeddings to float64 and L2-normalize (important for face vectors)
embeddings = np.asarray(embeddings, dtype=np.float64)
embeddings = normalize(embeddings, norm='l2')

def make_precomputed_distances(X, metric):
    """Return C-contiguous float64 pairwise distance matrix for HDBSCAN precomputed mode."""
    D = pairwise_distances(X, metric=metric, n_jobs=-1)
    return np.ascontiguousarray(D, dtype=np.float64)

def objective(trial):
    # --- Search space (practical ranges) ---
    min_cluster_size = trial.suggest_int('min_cluster_size', 2, 40)       # small->fine, large->coarse
    min_samples = trial.suggest_int('min_samples', 1, 40)                 # noise sensitivity
    metric = trial.suggest_categorical('metric', ['euclidean', 'cosine'])
    cluster_selection_method = trial.suggest_categorical('cluster_selection_method', ['eom', 'leaf'])
    alpha = trial.suggest_float('alpha', 0.5, 2.0)                        # hierarchy stability
    cluster_selection_epsilon = trial.suggest_float('cluster_selection_epsilon', 0.0, 0.2)
    leaf_size = trial.suggest_int('leaf_size', 20, 60)                    # influences BallTree / KDTree
    allow_single_cluster = trial.suggest_categorical('allow_single_cluster', [True, False])
    gen_min_span_tree = trial.suggest_categorical('gen_min_span_tree', [False])  # keep False for speed

    # Protect against degenerate combos quickly
    if min_samples > min_cluster_size:
        # avoid useless combinations where min_samples > min_cluster_size (rarely useful)
        return -1

    # Build clusterer and fit (handle cosine via precomputed distances)
    try:
        if metric == 'cosine':
            # Precompute cosine distances as float64 contiguous
            dists = make_precomputed_distances(embeddings, 'cosine')
            clusterer = hdbscan.HDBSCAN(
                min_cluster_size=min_cluster_size,
                min_samples=min_samples,
                metric='precomputed',
                cluster_selection_method=cluster_selection_method,
                alpha=alpha,
                cluster_selection_epsilon=cluster_selection_epsilon,
                leaf_size=leaf_size,
                allow_single_cluster=allow_single_cluster,
                gen_min_span_tree=gen_min_span_tree
            )
            labels = clusterer.fit_predict(dists)
        else:
            # euclidean path: ensure embeddings are float64 contiguous
            X = np.ascontiguousarray(embeddings, dtype=np.float64)
            clusterer = hdbscan.HDBSCAN(
                min_cluster_size=min_cluster_size,
                min_samples=min_samples,
                metric='euclidean',
                cluster_selection_method=cluster_selection_method,
                alpha=alpha,
                cluster_selection_epsilon=cluster_selection_epsilon,
                leaf_size=leaf_size,
                allow_single_cluster=allow_single_cluster,
                gen_min_span_tree=gen_min_span_tree
            )
            labels = clusterer.fit_predict(X)

    except Exception as e:
        # Return a poor score so Optuna moves on (logging optional)
        trial.set_user_attr("error", str(e))
        return -1

    # If only noise or single cluster — bad
    if len(set(labels)) <= 1 or np.all(labels == -1):
        return -1

    # Compute metrics using valid (non-noise) points
    try:
        labels_arr = np.asarray(labels)
        mask = labels_arr != -1
        valid_X = embeddings[mask]
        valid_labels = labels_arr[mask]

        # Need at least 2 clusters and some minimum points
        if len(set(valid_labels)) <= 1 or len(valid_X) < 10:
            return -1

        # Use euclidean silhouette for stability (works on normalized embeddings)
        sil = silhouette_score(valid_X, valid_labels, metric='euclidean')
        db = davies_bouldin_score(valid_X, valid_labels)
        score = float(sil - 0.2 * db)
        # record cluster count for analysis
        trial.set_user_attr("n_clusters", int(len(set(valid_labels))))
        trial.set_user_attr("n_noise", int(np.sum(labels_arr == -1)))
        return score
    except Exception as e:
        trial.set_user_attr("error_eval", str(e))
        return -1

# --- Run study ---
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("\n✅ BEST PARAMETERS FOUND:")
print(study.best_trial.params)
print("Extra attrs:", study.best_trial.user_attrs)


In [ ]:
# ============================================
# ⚙️ PRO-LEVEL FACE CLUSTERING PIPELINE (FIXED + OPTIMIZED PARAMS)
# ============================================

import os, cv2, torch, shutil, json, numpy as np
from pathlib import Path
from tqdm import tqdm
from insightface.app import FaceAnalysis
from PIL import Image, ImageDraw, ImageFont
from collections import defaultdict
import hdbscan
import math
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import pairwise_distances

# ---------- CONFIG ----------
INPUT_DIR = '/content/drive/MyDrive/MyDrive/input_photos'
OUTPUT_DIR = '/content/output_pro'
MAX_SIZE = 1600
MIN_FACE_SIZE = 80
MIN_DET_SCORE = 0.85
THUMB_SIZE = 160
PADDING = 5
SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)

# ---------- IMAGE PREPROCESSING ----------
def preprocess_image(img):
    """Resize + CLAHE grayscale + convert back to BGR"""
    h, w = img.shape[:2]
    if max(h, w) > MAX_SIZE:
        scale = MAX_SIZE / max(h, w)
        img = cv2.resize(img, (int(w * scale), int(h * scale)))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    gray = clahe.apply(gray)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

def list_images(folder):
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
    return sorted([str(p) for p in Path(folder).rglob('*') if p.suffix.lower() in exts])

def read_image(path):
    # robust file read that supports Unicode paths
    img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Unable to read {path}")
    return preprocess_image(img)

# ---------- FACE MODEL ----------
print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
face_app = FaceAnalysis(
    name='buffalo_l',
    providers=['CUDAExecutionProvider' if torch.cuda.is_available() else 'CPUExecutionProvider']
)
face_app.prepare(ctx_id=0 if torch.cuda.is_available() else -1)

# ---------- DETECT FACES & EMBEDDINGS ----------
image_paths = list_images(INPUT_DIR)
all_embeddings, meta = [], []

for img_path in tqdm(image_paths, desc="Detecting Faces"):
    try:
        img = read_image(img_path)
        faces = face_app.get(img)
        if not faces:
            continue
        for f in faces:
            x1, y1, x2, y2 = [int(v) for v in f.bbox]
            if (x2 - x1) < MIN_FACE_SIZE or (y2 - y1) < MIN_FACE_SIZE:
                continue
            if hasattr(f, 'det_score') and f.det_score < MIN_DET_SCORE:
                continue
            emb = f.embedding
            # guard: skip zero vectors
            if np.linalg.norm(emb) == 0:
                continue
            emb = emb / np.linalg.norm(emb)
            all_embeddings.append(emb.astype(np.float32))  # keep float32 for storage; convert later
            meta.append({
                'img': img_path,
                'bbox': [x1, y1, x2, y2],
                'embedding_idx': len(all_embeddings) - 1
            })
    except Exception as e:
        print(f"❌ {img_path} {e}")

if not all_embeddings:
    raise SystemExit("No faces detected!")

print(f"✅ Total faces detected: {len(all_embeddings)}")

# ---------- PREPARE EMBEDDINGS ----------
# convert to numpy array, L2-normalize and ensure float64 contiguous for HDBSCAN
X = np.vstack(all_embeddings)
X = np.asarray(X, dtype=np.float64)
X = normalize(X, norm='l2')
X = np.ascontiguousarray(X, dtype=np.float64)

# ---------- HDBSCAN WITH OPTIMAL HYPERPARAMETERS ----------
# Best params found by Optuna
best_params = {
    'min_cluster_size': 4,
    'min_samples': 3,
    'metric': 'cosine',  # we're going to safely handle this by precomputing distances
    'cluster_selection_method': 'eom',
    'alpha': 0.6627752394770419,
    'cluster_selection_epsilon': 0.1874627871474641,
    'leaf_size': 42,
    'allow_single_cluster': True,
    'gen_min_span_tree': False
}

print("\n🚀 Running HDBSCAN with best params:", best_params)

# If metric == 'cosine' we must pass a precomputed distance matrix (float64 contiguous)
if best_params['metric'] == 'cosine':
    # warn if dataset is large
    n = X.shape[0]
    if n > 8000:
        print(f"⚠ Warning: computing full pairwise distances for {n} samples may be heavy (O(n^2) memory).")
    # compute pairwise cosine distances (float64)
    D = pairwise_distances(X, metric='cosine', n_jobs=-1)
    D = np.ascontiguousarray(D, dtype=np.float64)

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=best_params['min_cluster_size'],
        min_samples=best_params['min_samples'],
        metric='precomputed',
        cluster_selection_method=best_params['cluster_selection_method'],
        alpha=best_params['alpha'],
        cluster_selection_epsilon=best_params['cluster_selection_epsilon'],
        leaf_size=best_params['leaf_size'],
        allow_single_cluster=best_params['allow_single_cluster'],
        gen_min_span_tree=best_params['gen_min_span_tree']
    )
    labels = clusterer.fit_predict(D)
else:
    # euclidean path (fast)
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=best_params['min_cluster_size'],
        min_samples=best_params['min_samples'],
        metric='euclidean',
        cluster_selection_method=best_params['cluster_selection_method'],
        alpha=best_params['alpha'],
        cluster_selection_epsilon=best_params['cluster_selection_epsilon'],
        leaf_size=best_params['leaf_size'],
        allow_single_cluster=best_params['allow_single_cluster'],
        gen_min_span_tree=best_params['gen_min_span_tree']
    )
    labels = clusterer.fit_predict(X)

# ---------- POSTPROCESS & SAVE ----------
labels = np.asarray(labels)
n_clusters = len([l for l in set(labels) if l != -1])
n_noise = int((labels == -1).sum())
print(f"✅ Clustering complete — {n_clusters} clusters, {n_noise} noise faces")

# Map clusters to person ids (person_01, person_02, ...)
unique_labels = sorted([l for l in set(labels) if l != -1])
cluster_to_person = {lbl: f"person_{i+1:02d}" for i, lbl in enumerate(unique_labels)}
person_assignments = ['unknown' if l == -1 else cluster_to_person[l] for l in labels]

# Create output folders
for pid in cluster_to_person.values():
    os.makedirs(os.path.join(OUTPUT_DIR, pid, 'solo'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, pid, 'group'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'unknown'), exist_ok=True)

# Save per-face cropped images into cluster folders (and gather per-image info)
cluster_dirs = {}
for idx, lbl in enumerate(labels):
    pid = 'unknown' if lbl == -1 else cluster_to_person[lbl]
    # ensure directory exists
    if lbl == -1:
        save_dir = os.path.join(OUTPUT_DIR, 'unknown')
    else:
        # store faces (cropped) inside person folder under 'faces' subfolder for clarity
        save_dir = os.path.join(OUTPUT_DIR, pid, 'faces')
        os.makedirs(save_dir, exist_ok=True)

    m = meta[idx]
    try:
        img = cv2.imdecode(np.fromfile(m['img'], dtype=np.uint8), cv2.IMREAD_COLOR)
        x1, y1, x2, y2 = m['bbox']
        # safety clamp bbox inside image
        h, w = img.shape[:2]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        face_crop = img[y1:y2, x1:x2]
        save_path = os.path.join(save_dir, f"{Path(m['img']).stem}_{idx}.jpg")
        cv2.imwrite(save_path, face_crop)
    except Exception as e:
        print(f"❌ Error saving face idx {idx} from {m['img']}: {e}")

# ---------- SOLO/GROUP ASSIGNMENT ----------
image_to_persons = defaultdict(list)
image_to_face_count = defaultdict(int)

for m, person in zip(meta, person_assignments):
    image_to_persons[m['img']].append(person)
    image_to_face_count[m['img']] += 1

for img, persons in tqdm(image_to_persons.items(), desc="Classifying Solo/Group"):
    face_count = image_to_face_count[img]
    unique_known = set([p for p in persons if p != 'unknown'])

    if face_count == 0 or not unique_known:
        try:
            shutil.copy2(img, os.path.join(OUTPUT_DIR, 'unknown', Path(img).name))
        except Exception:
            pass
        continue

    if face_count == 1:
        solo_pid = list(unique_known)[0]
        try:
            shutil.copy2(img, os.path.join(OUTPUT_DIR, solo_pid, 'solo', Path(img).name))
        except Exception:
            pass
        continue

    for pid in unique_known:
        try:
            shutil.copy2(img, os.path.join(OUTPUT_DIR, pid, 'group', Path(img).name))
        except Exception:
            pass

print("✅ Solo/Group classification done!")

# ---------- SAVE METADATA ----------
persons_meta = {}
for pid in cluster_to_person.values():
    solo_dir = Path(OUTPUT_DIR)/pid/'solo'
    group_dir = Path(OUTPUT_DIR)/pid/'group'
    faces_dir = Path(OUTPUT_DIR)/pid/'faces'
    persons_meta[pid] = {
        'solo_images': sorted([p.name for p in solo_dir.glob('*')]) if solo_dir.exists() else [],
        'group_images': sorted([p.name for p in group_dir.glob('*')]) if group_dir.exists() else [],
        'face_crops': sorted([p.name for p in faces_dir.glob('*')]) if faces_dir.exists() else []
    }

with open(Path(OUTPUT_DIR)/'persons_meta.json', 'w') as f:
    json.dump(persons_meta, f, indent=2)

# Save cluster summary
summary = {
    'best_hyperparameters': best_params,
    'n_clusters': int(n_clusters),
    'n_noise': int(n_noise)
}
with open(Path(OUTPUT_DIR)/'cluster_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("✅ Metadata saved.")
print(f"📊 Summary: {json.dumps(summary, indent=2)}")

# ---------- VISUAL REPORT WITH BOUNDING BOXES ----------
def display_thumbnail_grid(images, title="", thumb_size=THUMB_SIZE, padding=PADDING):
    if not images:
        print(f"No images for {title}")
        return
    n = len(images)
    cols = min(n, 5)
    rows = math.ceil(n / cols)
    grid_w = cols * thumb_size + (cols - 1) * padding
    grid_h = rows * thumb_size + (rows - 1) * padding
    grid_img = Image.new('RGB', (grid_w, grid_h), (255,255,255))
    for idx, img_path in enumerate(images):
        try:
            img = Image.open(img_path).convert('RGB')
            img.thumbnail((thumb_size, thumb_size))
            x = (idx % cols) * (thumb_size + padding)
            y = (idx // cols) * (thumb_size + padding)
            grid_img.paste(img, (x, y))
        except Exception:
            pass
    print(f"🔹 {title} ({len(images)} images)")
    display(grid_img)

# Display thumbnails for each person
for pid, data in persons_meta.items():
    solo_paths = [Path(OUTPUT_DIR)/pid/'solo'/name for name in data['solo_images']]
    group_paths = [Path(OUTPUT_DIR)/pid/'group'/name for name in data['group_images']]

    display_thumbnail_grid(solo_paths, title=f"{pid} - SOLO")
    display_thumbnail_grid(group_paths, title=f"{pid} - GROUP")


In [ ]:
# ============================================
# 📊 CLUSTERING EVALUATION METRICS
# ============================================

from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import numpy as np

# Convert to numpy arrays
labels = np.array(labels)
embeddings = np.array(embeddings)

# Filter out noise (-1) for metric computations
mask = labels != -1
valid_embeddings = embeddings[mask]
valid_labels = labels[mask]

if len(set(valid_labels)) > 1 and len(valid_embeddings) > 10:
    sil_score = silhouette_score(valid_embeddings, valid_labels, metric='euclidean')
    ch_score = calinski_harabasz_score(valid_embeddings, valid_labels)
    db_score = davies_bouldin_score(valid_embeddings, valid_labels)

    print("==== 🧠 CLUSTERING QUALITY METRICS ====")
    print(f"Silhouette Score          : {sil_score:.4f}  (Higher = Better)")
    print(f"Calinski-Harabasz Index   : {ch_score:.4f}  (Higher = Better)")
    print(f"Davies-Bouldin Index      : {db_score:.4f}  (Lower = Better)")
    print(f"Detected Clusters (no noise): {len(set(valid_labels))}")
    print(f"Noise Faces (-1 label): {np.sum(labels==-1)}")
else:
    print("⚠ Not enough valid clusters for evaluation.")